# Big Data Project: Wetter- und Luftqualitaetsdaten

## Einleitung

Dieses Projekt sammelt Wetter- und Luftqualitaetsdaten, speichert die Rohdaten lokal als JSON und legt sie zusaetzlich in MongoDB ab.

**Ziel des ersten Setups:**
- Python-Abhaengigkeiten installieren
- Projektordner vorbereiten
- `.env` und MongoDB-Verbindung pruefen
- das Projekt mit `src.main` starten

**Aktueller Ablauf im Projekt:**
1. Wetterdaten laden
2. Luftqualitaetsdaten laden
3. Rohdaten in `data/raw/...` speichern
4. Daten in MongoDB-Collections ablegen


## Setup

Vor dem Start bitte sicherstellen:
- Python 3 und ein aktives Projekt-Environment
- Docker bzw. Docker Desktop laeuft
- `docker compose` ist verfuegbar
- die folgende Setup-Zelle wird vor `src.main` ausgefuehrt

Die Setup-Zelle:
1. wechselt ins Projektverzeichnis
2. legt `.env` aus `.env.example` an, falls noetig
3. laedt die benoetigten Umgebungsvariablen
4. prueft Docker
5. startet den MongoDB-Container aus `docker-compose.yml`

Verwendete Standardwerte fuer `.env`:

```env
MONGO_URI=mongodb://localhost:27017/
MONGO_DB=big_data_weather_airpollution
WEATHER_API_KEY=
WEATHER_USE_MOCK=true
WEATHER_LAT=48.2082
WEATHER_LON=16.3738
WEATHER_UNITS=metric
WEATHER_LANG=de
WEATHER_EXCLUDE=minutely,alerts
AIR_QUALITY_API_KEY=
OPENAQ_LOCATION_ID=8118
```


In [22]:
%pip install -r requirements.txt


Note: you may need to restart the kernel to use updated packages.


**Hinweis:** Wenn `docker` oder `docker compose` nicht gefunden wird, bitte sicherstellen, dass Docker Desktop installiert und gestartet ist. Unter Linux muss eventuell der Docker-Daemon mit `sudo systemctl start docker` gestartet werden.

In [23]:
import subprocess

subprocess.run(
    ["docker", "ps"],
    check=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
print("Docker ist erreichbar.")

Docker ist erreichbar.


In [24]:
from pathlib import Path
import os
import shutil
import subprocess

from dotenv import dotenv_values

project_dir = Path.cwd()
os.chdir(project_dir)
print(f"Arbeitsverzeichnis: {project_dir}")

env_example = project_dir / ".env.example"
env_file = project_dir / ".env"

if not env_file.exists():
    shutil.copyfile(env_example, env_file)
    print(".env aus .env.example erstellt")
else:
    print(".env bereits vorhanden")

for key, value in dotenv_values(env_file).items():
    if value is not None:
        os.environ[key] = value

print(f"MONGO_URI={os.environ['MONGO_URI']}")
print(f"MONGO_DB={os.environ['MONGO_DB']}")

def run_command(command: list[str], success_message: str):
    result = subprocess.run(command, cwd=project_dir, text=True, capture_output=True)
    if result.returncode != 0:
        message = result.stderr.strip() or result.stdout.strip() or "Unbekannter Fehler"
        raise RuntimeError(f"Fehler bei {' '.join(command)}:\n{message}")
    print(success_message)

run_command(["docker", "compose", "up", "-d", "mongodb"], "MongoDB-Container laeuft.")


Arbeitsverzeichnis: C:\Users\aylin\DataspellProjects\BigData1
.env bereits vorhanden
MONGO_URI=mongodb://localhost:27017/
MONGO_DB=big_data_weather_airpollution
MongoDB-Container laeuft.


In [25]:
from pathlib import Path

folders = [
    Path("data/raw/weather"),
    Path("data/raw/air_quality"),
    Path("data/processed"),
]

for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)
    print(f"Ordner bereit: {folder}")


Ordner bereit: data\raw\weather
Ordner bereit: data\raw\air_quality
Ordner bereit: data\processed


In [26]:
from pathlib import Path

env_path = Path(".env")

if env_path.exists():
    print(".env gefunden")
    from dotenv import dotenv_values
    values = dotenv_values(env_path)
    for key in [
        "MONGO_URI",
        "MONGO_DB",
        "WEATHER_API_KEY",
        "WEATHER_USE_MOCK",
        "WEATHER_LAT",
        "WEATHER_LON",
        "WEATHER_UNITS",
        "WEATHER_LANG",
        "WEATHER_EXCLUDE",
        "AIR_QUALITY_API_KEY",
        "OPENAQ_LOCATION_ID",
    ]:
        value = values.get(key)
        if key.endswith("API_KEY"):
            state = "gesetzt" if value else "leer"
            print(f"{key}={state}")
        elif value is not None:
            print(f"{key}={value}")
else:
    print("Keine .env gefunden. Lege bitte eine Datei mit folgendem Inhalt an:")
    print("MONGO_URI=mongodb://localhost:27017/")
    print("MONGO_DB=big_data_weather_airpollution")
    print("WEATHER_API_KEY=")
    print("WEATHER_USE_MOCK=true")
    print("WEATHER_LAT=48.2082")
    print("WEATHER_LON=16.3738")
    print("WEATHER_UNITS=metric")
    print("WEATHER_LANG=de")
    print("WEATHER_EXCLUDE=minutely,alerts")
    print("AIR_QUALITY_API_KEY=")
    print("OPENAQ_LOCATION_ID=8118")


.env gefunden
MONGO_URI=mongodb://localhost:27017/
MONGO_DB=big_data_weather_airpollution
WEATHER_API_KEY=gesetzt
AIR_QUALITY_API_KEY=gesetzt


## Projektstart

Die naechste Zelle startet den aktuellen Einstiegspunkt `src.main` mit demselben Python-Interpreter wie das Notebook. Dabei werden Beispiel-Datensaetze erzeugt, lokal gespeichert und in MongoDB geschrieben.


In [27]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "src.main"], cwd=project_dir, check=True)


CompletedProcess(args=['C:\\Users\\aylin\\DataspellProjects\\BigData1\\.venv\\Scripts\\python.exe', '-m', 'src.main'], returncode=0)

## Aufraeumen und herunterfahren

Wenn du am Ende wieder ein sauberes System haben willst, kannst du zuerst den MongoDB-Container samt Compose-Ressourcen stoppen. Das Beenden von Docker Desktop ist optional und betrifft alle laufenden Docker-Container.


In [28]:
import subprocess

subprocess.run(["docker", "compose", "down"], cwd=project_dir, check=True)
print("MongoDB-Container und Compose-Ressourcen wurden gestoppt.")

# Optional unter Linux mit Docker Desktop als User-Service:
# subprocess.run(["systemctl", "--user", "stop", "docker-desktop"], check=True)
# print("Docker Desktop wurde gestoppt.")


MongoDB-Container und Compose-Ressourcen wurden gestoppt.
